In [3]:
import influxdb_client
from dotenv import load_dotenv
import os
from datetime import datetime, timezone, timedelta
import sys
sys.path.append('/home/xinqiao/new_mount/gaussian_sampler/Gaussian_Sampler')

def initialize_client():
    load_dotenv()
    
    org = os.getenv("ORG")
    token = os.getenv("API_KEY")
    url = os.getenv("DB_URL")
    
    client = influxdb_client.InfluxDBClient(
        url=url,
        token=token,
        org=org
    )
    
    return client


def get_sensor_data(start_date: datetime, end_date: datetime, sensor: str, 
                      celsius=False, fahrenheit=False, pressure=False, humidity=False):
    
    fields_to_query = []
    if celsius:
        fields_to_query.append("celcius")
    if fahrenheit:
        fields_to_query.append("fahrenheit")
    if pressure:
        fields_to_query.append("pressure")
    if humidity:
        fields_to_query.append("humidity")

    if not fields_to_query:
        print("No data fields selected. Returning no data.")
        return []

    flux_field_filter = " or ".join([f'r._field == "{field}"' for field in fields_to_query])

    duration = end_date - start_date
    duration_in_hours = duration.total_seconds() / 3600

    window_period = None

    if duration_in_hours > 336:
        window_period = "90m"
    elif duration_in_hours > 168:
        window_period = "30m"
    elif duration_in_hours > 72:
        window_period = "10m"
    elif duration_in_hours > 24:
        window_period = "5m"

    aggregate_string = ""
    if window_period:
        aggregate_string = f'''
          |> aggregateWindow(every: {window_period}, fn: mean, createEmpty: false)
        '''
    
    start = start_date.isoformat()
    end = end_date.isoformat()

    query = f'''
        from(bucket: "{os.getenv("BUCKET")}")
          |> range(start: {start}, stop: {end})
          |> filter(fn: (r) => r.topic == "sensor/{sensor}")
          |> filter(fn: (r) => {flux_field_filter})
          {aggregate_string}
    '''
    
    client = initialize_client()
    query_api = client.query_api()
    result_tables = query_api.query(org=os.getenv("ORG"), query=query)
    client.close()

    results = []
    for table in result_tables:
        for record in table.records:
            results.append({
                "time": record.get_time().isoformat(),
                "field": record.get_field(),
                "value": record.get_value()
            })
            
    return results

In [4]:

end_utc = datetime.now(timezone.utc)
start_utc = end_utc - timedelta(minutes=30)

res = get_sensor_data(start_utc, end_utc, "lab", celsius=True);

In [5]:
res


[{'time': '2026-03-14T23:08:42+00:00', 'field': 'celcius', 'value': 21.79},
 {'time': '2026-03-14T23:09:12+00:00', 'field': 'celcius', 'value': 21.8},
 {'time': '2026-03-14T23:09:42+00:00', 'field': 'celcius', 'value': 21.82},
 {'time': '2026-03-14T23:10:12+00:00', 'field': 'celcius', 'value': 21.84},
 {'time': '2026-03-14T23:10:42+00:00', 'field': 'celcius', 'value': 21.86},
 {'time': '2026-03-14T23:11:12+00:00', 'field': 'celcius', 'value': 21.88},
 {'time': '2026-03-14T23:11:42+00:00', 'field': 'celcius', 'value': 21.9},
 {'time': '2026-03-14T23:12:12+00:00', 'field': 'celcius', 'value': 21.9},
 {'time': '2026-03-14T23:12:42+00:00', 'field': 'celcius', 'value': 21.9},
 {'time': '2026-03-14T23:13:12+00:00', 'field': 'celcius', 'value': 21.9},
 {'time': '2026-03-14T23:13:42+00:00', 'field': 'celcius', 'value': 21.9},
 {'time': '2026-03-14T23:14:12+00:00', 'field': 'celcius', 'value': 21.91},
 {'time': '2026-03-14T23:14:42+00:00', 'field': 'celcius', 'value': 21.94},
 {'time': '2026-03